# **05 - Model Comparison**

This notebook compares the XGBoost and LSTM models on a shared, identifier-matched test set, since each model's own train/test split was drawn independently in `03_xgboost.ipynb` and `04_lstm.ipynb` and cannot be assumed to align by row position alone. Predictions are merged on matchup identifiers, with an exlpicit check that both models' outcome labels agree on every shared row before any comparison is trusted. AUC and log-loss are computed on this shared subset, and an ROC curve is plotted to visualize the two models across the full range of decision thresholds rather than a single operating point.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, roc_curve, log_loss
import matplotlib.pyplot as plt

xgb_preds = pd.read_parquet("xgboost_predictions.parquet")
lstm_preds = pd.read_parquet("lstm_predictions.parquet")

comparison = xgb_preds.merge(lstm_preds, on=["gameId", "playId", "nflId", "rusher_nflId"], suffixes=("_xgb", "_lstm"),)
print(f"XGBoost test set: {len(xgb_preds):,}")
print(f"LSTM test set: {len(lstm_preds):,}")
print(f"Matchups present in both test sets: {len(comparison):,}")

In [ ]:
y_true = comparison["blocker_lost_actual_xgb"]
auc_xgb = roc_auc_score(y_true, comparison["blocker_lost_predicted_proba_xgb"])
auc_lstm = roc_auc_score(y_true, comparison["blocker_lost_predicted_proba_lstm"])
ll_xgb = log_loss(y_true, comparison["blocker_lost_predicted_proba_xgb"])
ll_lstm = log_loss(y_true, comparison["blocker_lost_predicted_proba_lstm"])
summary = pd.DataFrame({
    "model": ["XGBoost", "LSTM"],
    "AUC": [auc_xgb, auc_lstm],
    "log_loss": [ll_xgb, ll_lstm],
})
summary

In [ ]:
fpr_xgb, tpr_xgb, _ = roc_curve(y_true, comparison["blocker_lost_predicted_proba_xgb"])
fpr_lstm, tpr_lstm, _ = roc_curve(y_true, comparison["blocker_lost_predicted_proba_lstm"])

plt.plot(fpr_xgb, tpr_xgb, label = f"XGBoost (AUC = {auc_xgb: .3f})")
plt.plot(fpr_lstm, tpr_lstm, label = f"LSTM (AUC = {auc_lstm: .3f})")
plt.plot([0, 1], [0, 1], "k--", alpha = 0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost vs LSTM - ROC Comparison")
plt.legend()
plt.savefig("model_comparison_roc.png", dpi = 150, bbox_inches = "tight")
plt.show()

In [ ]:
comparison.to_parquet("model_comparison.parquet", index=False)
summary.to_csv("model_comparison_summary.csv", index=False)
print("Comparison saved.")